# ProbLog (probabilistic logic)

**Domain:** Symbolic AI & Logic  ·  **recommended addition**  ·  **runnable:** yes

A compact refresher on **ProbLog** — Prolog where facts and rules carry probabilities, and
queries return the *probability that an atom is true* by summing over all the possible worlds
a probabilistic program describes. It is the canonical **probabilistic logic programming (PLP)**
language: a clean marriage of logical rules and Bayesian inference.

## 1. What & Why

**What it is.** ProbLog extends Prolog with one idea: any fact (or rule head) may be annotated
with a probability, written `p::atom`. Each such *probabilistic fact* is an independent coin
flip — true with probability `p`, false otherwise. A program with `n` probabilistic facts
therefore defines a distribution over `2^n` **possible worlds** (one per truth assignment). The
probability of a query atom is the total probability mass of the worlds in which ordinary Prolog
derivation proves it.

**The problem it solves.** Plain Prolog gives you crisp yes/no logical consequence; Bayesian
networks give you probabilities but no relational structure, recursion, or rich rule logic.
ProbLog gives you *both* — you write declarative rules the way you would in Prolog, attach
probabilities to the uncertain bits, and get exact (or approximate) marginal and conditional
probabilities out. It subsumes Bayesian networks, Markov chains, and noisy-OR models as special
cases while letting you express things they can't: unbounded recursion, relational templates
over `person(X)`, structured background knowledge.

**When to reach for it.** Relational domains with uncertainty: link prediction in networks,
probabilistic knowledge graphs, gene/disease pathways, noisy sensor fusion over logical rules,
and any "Bayesian network but the structure is generated by rules" problem. Also for *learning*
rule probabilities from data (LFI — Learning From Interpretations). **When not to:** dense
continuous-variable models (use a PPL like PyMC/Stan), pure deterministic logic (use SWI-Prolog),
or problems where the number of relevant probabilistic facts explodes — exact inference is
#P-hard and the grounding can blow up.

## 2. Mental Model

> **A probabilistic program = a bag of independent coins + a Prolog rulebook. Flip every coin to
> get one *possible world*; run ordinary Prolog in it; the answer's probability is the share of
> worlds where Prolog proves it.**

```
  0.6::rain.        coin R: heads (rain) w.p. 0.6
  0.8::sprinkler.   coin S: heads (sprinkler) w.p. 0.8
  wet :- rain.      deterministic rules, no coins
  wet :- sprinkler.

  worlds (R,S):  TT .48 -> wet   TF .12 -> wet
                 FT .32 -> wet   FF .08 -> dry
  P(wet) = .48+.12+.32 = 0.92   (= 1 - (1-.6)(1-.8))
```

Because the two rules share the head `wet`, ProbLog automatically does the **noisy-OR**: it does
*not* double-count the `TT` world. Internally it compiles the proofs into a logic formula and
weights it (knowledge compilation to a d-DNNF / SDD), which is why it gets the inclusion-
exclusion right where a naive "add up the rule probabilities" would give `0.6+0.8 > 1`.

## 3. Key Concepts

| Term | What it means |
| --- | --- |
| **Probabilistic fact** | `p::f.` — `f` is an independent Bernoulli(`p`) atom. The atomic source of randomness. |
| **Possible world** | One joint truth assignment to all probabilistic facts; its probability is the product of the per-fact probabilities. |
| **Annotated disjunction (AD)** | `p1::a ; p2::b ; p3::c.` — a single mutually-exclusive choice (probabilities sum to ≤ 1, remainder = none). Models a die, a category pick. |
| **Probabilistic clause** | `p::head :- body.` — head holds w.p. `p` *given* the body proves. Sugar for a fresh probabilistic fact ANDed into the body. |
| **Query** | `query(atom).` — ask for `P(atom)`, marginalized over all worlds. Use a variable (`query(smokes(_))`) to get one row per ground instance. |
| **Evidence** | `evidence(atom, true/false).` — condition on observations; queries then return *posterior* `P(query \| evidence)`. |
| **Inference** | Exact via knowledge compilation (SDD/d-DNNF) — default; or approximate (sampling, k-best) for large/intractable programs. |
| **Noisy-OR** | Multiple rules with the same head combine as `1 - ∏(1 - p_i)`, handled automatically — a key reason to use ProbLog over hand-rolled BNs. |
| **LFI** | Learning From Interpretations: fit unknown `t(_)::` fact probabilities from a set of partial-observation examples (EM). |

## 4. Setup

Pure Python, no native compiler needed for the default exact solver (it ships a built-in SDD/
d-DNNF pipeline). One small wheel:

```bash
pip install problog
```

Optional extras: `problog install` pulls native SDD libraries for faster compilation on large
models, and a system C compiler helps for very big programs — neither is needed for the tiny
examples here. ProbLog also has a CLI (`problog model.pl`) if you'd rather keep models in `.pl`
files; below we drive it from Python with `PrologString`.

In [ ]:
# %pip install problog   # uncomment on a fresh kernel
import problog
from problog.version import version as problog_version
print('problog', problog_version)

## 5. Worked Examples

Three tiny, CPU-only models, each highlighting one capability: noisy-OR marginals, conditioning
on evidence (Bayes), and recursion over a relational network.

### Example 1 — Marginals & automatic noisy-OR (the sprinkler)

Two independent causes (`rain`, `sprinkler`) each make the grass `wet`. We query both the raw
fact and the derived atom; ProbLog combines the two `wet` rules correctly as `1-(1-.6)(1-.8)=0.92`
rather than naively adding the probabilities.

In [ ]:
from problog.program import PrologString
from problog import get_evaluatable

sprinkler = PrologString('''
    0.6::rain.
    0.8::sprinkler.

    wet :- rain.
    wet :- sprinkler.

    query(rain).
    query(wet).
''')

result = get_evaluatable().create_from(sprinkler).evaluate()
for atom, prob in result.items():
    print(f'{str(atom):10s} {prob:.4f}')

### Example 2 — Evidence & conditional inference (a noisy test)

A rare disease (prior 1%) and a test that is 90% sensitive and 5% false-positive. We `evidence`
a positive test and query the disease — ProbLog returns the Bayesian posterior
`P(disease | pos) = (.01·.9)/(.01·.9 + .99·.05) ≈ 0.154`. The two probabilistic clauses share
the head `pos_test`, modelling the true-positive and false-positive paths.

In [ ]:
medical = PrologString(r'''
    0.01::disease.
    0.90::pos_test :- disease.
    0.05::pos_test :- \+disease.

    evidence(pos_test, true).
    query(disease).
''')

post = get_evaluatable().create_from(medical).evaluate()
print('P(disease | positive test) =', round(list(post.values())[0], 4))

### Example 3 — Annotated disjunction + recursion (the smokers network)

The textbook PLP example. People are stressed (which makes them smoke), and friends *influence*
each other, so smoking propagates along the friendship graph — a **recursive** rule. We also show
an **annotated disjunction** for a fair three-sided die: exactly one face comes up. Querying with
a free variable (`query(smokes(_))`) returns one probability per person, computed over the whole
network of possible worlds.

In [ ]:
social = PrologString('''
    % --- annotated disjunction: exactly one face of a 3-sided die ---
    1/3::die(1); 1/3::die(2); 1/3::die(3).

    % --- recursive smokers network ---
    0.3::stress(X)        :- person(X).
    0.2::influences(X, Y) :- friend(X, Y).

    person(alice). person(bob). person(carl).
    friend(alice, bob). friend(bob, carl).

    smokes(X) :- stress(X).
    smokes(X) :- friend(Y, X), influences(Y, X), smokes(Y).

    query(die(_)).
    query(smokes(_)).
''')

for atom, prob in sorted(get_evaluatable().create_from(social).evaluate().items(), key=str):
    print(f'{str(atom):16s} {prob:.4f}')

Reading the output: each `die(k)` is `0.333`. `smokes(alice)` is just her stress prior `0.3`;
`bob` and `carl` are higher because they can *also* catch it down the friendship chain — exactly
the kind of relational, recursive uncertainty a plain Bayesian network can't express compactly.

## 6. Gotchas & Pitfalls

- **Probabilistic facts are independent.** `0.5::a.` and `0.5::b.` are two separate coins. If you
  need mutual exclusion or a shared cause, use an **annotated disjunction** or a common
  probabilistic ancestor — don't assume correlation appears on its own.
- **Same head ⇒ noisy-OR, not sum.** Multiple clauses for one atom combine as `1-∏(1-p_i)`. This
  is usually what you want, but it surprises people who expect the probabilities to add (and it's
  why two `0.6` rules give `0.84`, not `1.2`).
- **Negation needs `\+` and a probability semantics.** ProbLog uses the **distribution
  semantics**; programs must be sound (no probabilistic facts in negative cycles). Stratify your
  negation — cyclic `\+` over probabilistic atoms is rejected or ill-defined.
- **Grounding blow-up.** Inference grounds the program first. Large domains or many
  interdependent probabilistic facts make the compiled circuit explode (#P-hard). Switch to
  approximate inference (sampling / k-best) or shrink the relevant program before blaming speed.
- **`query(p(_))` enumerates ground instances** — make sure the predicate is range-restricted by
  facts, or you'll get nothing (unbound) or a combinatorial fan-out.
- **Probabilities must be valid.** In an annotated disjunction the weights must sum to ≤ 1; a
  `p::` outside `[0,1]` is an error. `1/3` works (ProbLog evaluates the arithmetic).
- **Results dict keys are `Term` objects**, not strings — `str(atom)` to print, and order is not
  guaranteed, so sort if you need stable output.

## 7. When to Use vs Alternatives

| Option | Use it when… | Trade-off vs ProbLog |
| --- | --- | --- |
| **ProbLog** | Relational + recursive structure with uncertainty; you want declarative rules and exact marginals/posteriors; learning rule weights from interpretations. | Exact inference is #P-hard; grounding can blow up; discrete-only. |
| **Bayesian network** (pgmpy, BNs) | Fixed, modest set of discrete variables with a known DAG. | No recursion, no relational templates; you hand-build the structure ProbLog would generate from rules. |
| **General PPL** (PyMC, Stan, NumPyro) | Continuous variables, hierarchical/Bayesian regression, gradient-based MCMC/VI. | Great at continuous + sampling, weak at crisp logical rules and discrete combinatorial structure. |
| **Plain Prolog / ASP** (SWI-Prolog, Clingo) | Purely logical reasoning, hard constraints, no probabilities. | No uncertainty quantification — ProbLog reduces to Prolog when all `p=1`. |
| **Other PLP** (PRISM, cplint, Pyro/ProbLog-style) | Need a specific learning algorithm, continuous extensions (cplint's `hybrid`), or deep-learning hooks (**DeepProbLog** for neural predicates). | DeepProbLog adds neural networks on top of exactly this engine; reach for it when facts come from a perceptual model. |

Rule of thumb: **discrete + relational + rules + uncertainty → ProbLog.** Add neural perception →
DeepProbLog. Go continuous → a sampling PPL. Drop the uncertainty → Prolog/ASP.

## 8. Resources

- **Official site & docs** — https://dtai.cs.kuleuven.be/problog/ (tutorials, online editor, API).
- **GitHub (source, issues, examples)** — https://github.com/ML-KULeuven/problog
- **Foundational paper** — De Raedt, Kimmig, Toivonen, *"ProbLog: A Probabilistic Prolog and Its
  Application in Link Discovery"* (IJCAI 2007): https://dtai.cs.kuleuven.be/problog/publications.html
- **Distribution semantics / PLP survey** — Riguzzi, *Foundations of Probabilistic Logic
  Programming* (open access): https://www.riverpublishers.com/book_details.php?book_id=773
- **DeepProbLog** (neural-symbolic extension) — https://github.com/ML-KULeuven/deepproblog
- Related notebooks in this collection: **swi-prolog** (the deterministic base), **answer-set-
  programming** (logical alternative), **z3-smt** (constraint solving).

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def marginal(probs, holds, evidence=None):
    """The probability that `holds` is true, conditioned on `evidence`."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE